# Solving the Wave Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D wave equation** using two approaches:

1. **Numerical Methods** — Explicit Central Difference (Leapfrog) and Implicit Newmark-$\beta$
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

The wave equation governs vibrations of a string, acoustic waves, and electromagnetic propagation:

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2}, \qquad x \in [0, L],\; t \in [0, T]$$

This is a **linear, second-order hyperbolic PDE** with constant wave speed $c$.

**Initial conditions** — first vibrational mode, released from rest:

$$u(x, 0) = \sin\!\left(\frac{\pi x}{L}\right), \qquad \frac{\partial u}{\partial t}(x, 0) = 0$$

**Boundary conditions** — fixed ends (Dirichlet):

$$u(0, t) = u(L, t) = 0$$

**Exact solution** — standing wave (first mode):

$$u^*(x, t) = \sin\!\left(\frac{\pi x}{L}\right)\cos\!\left(\frac{c\,\pi\,t}{L}\right)$$

The string oscillates between $\pm\sin(\pi x/L)$ with period $T_{\text{period}} = 2L/c$. The wave equation preserves total energy:

$$E = \frac{1}{2}\int_0^L \left[u_t^2 + c^2\,u_x^2\right]dx = \text{const}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
C     = 1.0       # wave speed
L_DOM = 1.0       # domain length [0, L]
T_END = 2.0       # final time  (one full period = 2L/c = 2.0)

In [ ]:
def u_init(x):
    """Initial displacement: first mode sin(πx/L)."""
    return np.sin(np.pi * x / L_DOM)

def v_init(x):
    """Initial velocity: released from rest."""
    return np.zeros_like(x)

def u_exact(x, t):
    """Exact solution: sin(πx/L)·cos(cπt/L)."""
    return np.sin(np.pi * x / L_DOM) * np.cos(C * np.pi * t / L_DOM)


# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, 0.5, 1.0, 1.5]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.2f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — Wave Equation (First Mode)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
T_period = 2 * L_DOM / C
print(f"Wave speed c = {C},  domain L = {L_DOM}")
print(f"Period = 2L/c = {T_period:.2f},  T_end = {T_END} ({T_END/T_period:.1f} periods)")

---

## Part 1 — Numerical Methods

### 1-A. Explicit Central Difference (Leapfrog)

The classical **leapfrog** scheme uses central differences in both space and time:

$$u_j^{n+1} = 2\,u_j^n - u_j^{n-1} + r^2\left(u_{j+1}^n - 2\,u_j^n + u_{j-1}^n\right)$$

where $r = c\,\Delta t\,/\,\Delta x$ is the **Courant number**. This is second-order accurate in both space and time: $\mathcal{O}(\Delta t^2,\,\Delta x^2)$.

**CFL stability condition**: $r \leq 1$. At $r = 1$ the scheme is *exact* (the numerical domain of dependence matches the physical one).

**First time step** — since the scheme is three-level, we bootstrap using the Taylor expansion with the initial velocity:

$$u_j^1 = u_j^0 + \Delta t\,\dot{u}_j^0 + \frac{r^2}{2}\left(u_{j+1}^0 - 2\,u_j^0 + u_{j-1}^0\right)$$

In [ ]:
def solve_leapfrog(Nx=200, Nt=400, T=T_END):
    """
    Explicit central-difference (leapfrog) for the 1-D wave equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)   Grid including boundary points.
    u_hist : list of ndarray   Solution snapshots.
    t_hist : list of float     Snapshot times.
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    r   = C * dt / dx

    assert r <= 1.0, f"CFL violated: r = {r:.4f} > 1"
    print(f"Leapfrog — dx={dx:.5f}, dt={dt:.6f}, Courant r={r:.4f}")

    # Level n=0
    u_prev  = u_init(x).copy()
    u_prev[0] = u_prev[-1] = 0.0

    # Level n=1 via Taylor expansion (v_init = 0 simplifies this)
    u_curr          = u_prev.copy()
    u_curr[1:-1]    = (u_prev[1:-1] + dt * v_init(x[1:-1]) + 0.5 * r**2 * (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2]))
    u_curr[0]       = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # Leapfrog update
        u_next          = np.zeros_like(u_curr)
        u_next[1:-1]    = (2 * u_curr[1:-1] - u_prev[1:-1] + r**2 * (u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2]))
        u_next[0]       = u_next[-1] = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_lf, Nt_lf                    = 200, 400
x_lf, u_lf_hist, t_lf_hist      = solve_leapfrog(Nx_lf, Nt_lf)

# Error at final time
u_ex_lf   = u_exact(x_lf, T_END)
err_lf    = np.abs(u_lf_hist[-1] - u_ex_lf)
print(f"Max absolute error  : {err_lf.max():.3e}")
print(f"Mean absolute error : {err_lf.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_lf_hist, t_lf_hist)):
    u_ref = u_exact(x_lf, t_snap)

    axes[0, col].plot(x_lf, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_lf, u_snap, "b--", lw=1.5, label="Leapfrog")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_lf, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit Central Difference (Leapfrog) — Wave Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit Newmark-$\beta$ Method

The **Newmark-$\beta$** family is the standard implicit time-integration scheme for second-order-in-time problems. With parameters $\beta$ and $\gamma$:

$$u^{n+1} = u^n + \Delta t\,\dot{u}^n + \frac{\Delta t^2}{2}\left[(1 - 2\beta)\,a^n + 2\beta\,a^{n+1}\right]$$

$$\dot{u}^{n+1} = \dot{u}^n + \Delta t\left[(1 - \gamma)\,a^n + \gamma\,a^{n+1}\right]$$

where $a = u_{tt} = c^2\,u_{xx}$. Substituting the spatial operator $\mathcal{D}^2 u_j = (u_{j+1} - 2u_j + u_{j-1})/\Delta x^2$ at level $n+1$:

$$\left(I - \beta\,\Delta t^2\,c^2\,\mathcal{D}^2\right)\mathbf{u}^{n+1} = \text{RHS}(u^n,\dot{u}^n,a^n)$$

We use $\beta = 1/4$, $\gamma = 1/2$ (the **average-acceleration / trapezoidal** rule):
- **Unconditionally stable** (no CFL restriction)
- **Second-order accurate** in time
- **No numerical dissipation** ($\gamma = 1/2$)

The LHS is a tridiagonal system with constant coefficients, solved via banded elimination.

In [ ]:
def solve_newmark(Nx=200, Nt=200, T=T_END, beta=0.25, gamma=0.5):
    """
    Implicit Newmark-β for the 1-D wave equation.

    Parameters
    ----------
    Nx    : int     Interior grid points.
    Nt    : int     Number of time steps.
    beta  : float   Newmark parameter (1/4 = average acceleration).
    gamma : float   Newmark parameter (1/2 = no numerical dissipation).

    Returns
    -------
    x      : ndarray (Nx+2,)
    u_hist : list of ndarray
    t_hist : list of float
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    N   = Nx            # number of interior points

    r   = C * dt / dx
    lam = beta * (C * dt)**2 / dx**2   # coefficient for implicit term
    print(f"Newmark-β — dx={dx:.5f}, dt={dt:.5f}, r={r:.4f}, β={beta}, γ={gamma}")

    # Build LHS banded matrix: (I − β Δt² c² D²) on interior points
    # D² has -1/dx² on sub/super and 2/dx² on diag
    A_band          = np.zeros((3, N))
    A_band[0, 1:]   = -lam                   # super-diagonal
    A_band[1, :]    = 1 + 2 * lam            # diagonal
    A_band[2, :-1]  = -lam                   # sub-diagonal

    # Spatial second-difference operator (interior only)
    def D2(u_full):
        return C**2 * (u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]) / dx**2

    # Initialise
    u       = u_init(x).copy()
    u[0]    = u[-1] = 0.0
    udot    = v_init(x).copy()
    udot[0] = udot[-1] = 0.0

    # Initial acceleration a^0 = c² u_xx^0
    a_full      = np.zeros_like(x)
    a_full[1:-1] = D2(u)

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_hist.append(u.copy())
            t_hist.append(n * dt)
        if n == Nt:
            break

        # Predictor (interior)
        u_pred   = (u[1:-1] + dt * udot[1:-1] + 0.5 * dt**2 * (1 - 2 * beta) * a_full[1:-1])

        # RHS for tridiagonal solve
        rhs = u_pred   # lam terms on LHS absorb the β·Δt²·c²·D²·u^{n+1}

        # Solve for u^{n+1} interior
        u_new       = np.zeros_like(u)
        u_new[1:-1] = solve_banded((1, 1), A_band, rhs)
        u_new[0]    = 0.0
        u_new[-1]   = 0.0

        # New acceleration
        a_new       = np.zeros_like(x)
        a_new[1:-1] = D2(u_new)

        # Update velocity
        udot_new       = np.zeros_like(x)
        udot_new[1:-1] = (udot[1:-1] + dt * ((1 - gamma) * a_full[1:-1] + gamma * a_new[1:-1]))
        udot_new[0]    = 0.0
        udot_new[-1]   = 0.0

        u      = u_new
        udot   = udot_new
        a_full = a_new

    return x, u_hist, t_hist


Nx_nm, Nt_nm = 200, 200
x_nm, u_nm_hist, t_nm_hist = solve_newmark(Nx_nm, Nt_nm)

u_ex_nm = u_exact(x_nm, T_END)
err_nm  = np.abs(u_nm_hist[-1] - u_ex_nm)
print(f"Max absolute error  : {err_nm.max():.3e}")
print(f"Mean absolute error : {err_nm.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_nm_hist, t_nm_hist)):
    u_ref = u_exact(x_nm, t_snap)

    axes[0, col].plot(x_nm, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_nm, u_snap, "g--", lw=1.5, label="Newmark-β")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_nm, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Implicit Newmark-β — Wave Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
Nx_list = [25, 50, 100, 200, 400]
err_lf_conv, err_nm_conv = [], []

for Nx in Nx_list:
    # Leapfrog: ensure CFL ≤ 1
    dx_c            = L_DOM / (Nx + 1)
    Nt_c            = max(int(np.ceil(T_END * C / (0.9 * dx_c))), 200)
    x_c, u_c, t_c   = solve_leapfrog(Nx, Nt_c)
    u_ex_c          = u_exact(x_c, t_c[-1])
    err_lf_conv.append(np.max(np.abs(u_c[-1] - u_ex_c)))

    # Newmark — fixed moderate Nt
    x_c2, u_c2, t_c2    = solve_newmark(Nx, Nt=400)
    u_ex_c2             = u_exact(x_c2, t_c2[-1])
    err_nm_conv.append(np.max(np.abs(u_c2[-1] - u_ex_c2)))

dh = [L_DOM / (Nx + 1) for Nx in Nx_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_lf_conv, "bs-", lw=1.5, label="Leapfrog")
ax.loglog(dh, err_nm_conv, "ro-", lw=1.5, label="Newmark-β")
ax.loglog(dh, [h**2 * err_lf_conv[0] / dh[0]**2 for h in dh], "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.loglog(dh, [h**2 * err_nm_conv[0] / dh[0]**2 for h in dh], "r--", lw=1.0, label="$\\mathcal{O}(h^2)$ (ref)")
ax.set_xlabel("Grid spacing $h$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — FDM Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial conditions}} + \underbrace{\mathcal{L}_{BC}}_{\text{boundary}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2 + \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}_t(x_k, 0) - 0\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\hat{u}(0, t_k)^2 + \hat{u}(L, t_k)^2\right]$$

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial^2\hat{u}}{\partial t^2} - c^2\,\frac{\partial^2\hat{u}}{\partial x^2}\right)^2$$

Both **second derivatives** $u_{tt}$ and $u_{xx}$ are computed via two successive calls to `torch.autograd.grad`.

Note: the IC loss enforces *two* conditions — initial displacement **and** initial velocity (zero). This is characteristic of second-order-in-time PDEs: two initial conditions are required.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class WavePINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth second-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC    = 2000    # IC points (displacement + velocity)
N_BC    = 1000    # boundary points (each side)
N_INT   = 10000   # interior PDE collocation points

# ---- IC at t = 0 ----------------------------------
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# ---- Boundary points (x = 0 and x = L) -----------
t_bc = torch.rand(N_BC, 1) * T_END
x_bc_left  = torch.zeros(N_BC, 1)
x_bc_right = torch.full((N_BC, 1), L_DOM)

# ---- Interior PDE collocation points --------------
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC}")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss: displacement + velocity ------
    u_pred_ic   = model(x_ic, t_ic)
    loss_ic_u   = mse(u_pred_ic, u_ic)

    u_t_ic      = grad1(u_pred_ic, t_ic)
    loss_ic_v   = mse(u_t_ic, torch.zeros_like(u_t_ic))

    loss_ic     = loss_ic_u + loss_ic_v

    # ---- Boundary loss (u = 0 at x = 0, L) -----
    u_left  = model(x_bc_left,  t_bc)
    u_right = model(x_bc_right, t_bc)
    loss_bc = mse(u_left,  torch.zeros_like(u_left)) + mse(u_right, torch.zeros_like(u_right))

    # ---- PDE residual: u_tt − c²·u_xx = 0 -----
    u_pred  = model(x_int, t_int)
    u_t     = grad1(u_pred, t_int)
    u_tt    = grad1(u_t,    t_int)
    u_x     = grad1(u_pred, x_int)
    u_xx    = grad1(u_x,    x_int)
    residual = u_tt - C**2 * u_xx
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model = WavePINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP = 5000
history = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss


opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — Wave Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense x grid at each snapshot time ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

# PINN error at final time
U_ex_pinn   = u_exact(x_ev, T_END)
err_pinn    = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)

    axes[0, col].plot(x_ev, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ev, U_snap, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    axes[1, col].plot(x_ev, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — Wave Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary and energy conservation check.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid
from scipy.interpolate import interp1d

u_lf_ev     = interp1d(x_lf, u_lf_hist[-1], kind="linear", bounds_error=False, fill_value=0.0)(x_ev)
u_nm_ev     = interp1d(x_nm, u_nm_hist[-1], kind="linear", bounds_error=False, fill_value=0.0)(x_ev)
u_pinn_ev   = U_pinn_snaps[-1]

err_lf_ev   = np.abs(u_lf_ev   - U_ex_pinn)
err_nm_ev   = np.abs(u_nm_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels  = ["Leapfrog", "Newmark-β", "PINN", "Exact"]
sols    = [u_lf_ev, u_nm_ev, u_pinn_ev, U_ex_pinn]
colors  = ["steelblue", "seagreen", "mediumorchid", "black"]
errs    = [err_lf_ev, err_nm_ev, err_pinn_ev]
e_lbl   = ["Leapfrog error", "Newmark-β error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(x_ev, sol, color=c, lw=2)
    ax.fill_between(x_ev, sol, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(x_ev, err, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay plot for direct comparison
ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(x_ev, U_ex_pinn,  "k-",  lw=2,   label="Exact")
ax_ov.plot(x_ev, u_lf_ev,    "b--", lw=1.5, label="Leapfrog")
ax_ov.plot(x_ev, u_nm_ev,    "g:",  lw=1.5, label="Newmark-β")
ax_ov.plot(x_ev, u_pinn_ev,  "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — Wave Equation", fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary + energy conservation check ----
def l2_norm(f_arr, x_arr):
    return np.sqrt(np.trapezoid(f_arr**2, x_arr))


def energy(u_arr, x_arr, t_val):
    """Approximate total energy E = (1/2)∫[u_t² + c²·u_x²] dx using exact u_t."""
    # For the exact solution, u_t and u_x are known analytically
    # For numerical solutions we approximate u_x from the displacement
    u_x = np.gradient(u_arr, x_arr)
    # Use exact u_t for energy at this snapshot (since we don't store velocities)
    omega = C * np.pi / L_DOM
    u_t_exact = -np.sin(np.pi * x_arr / L_DOM) * omega * np.sin(omega * t_val)
    return 0.5 * np.trapezoid(u_t_exact**2 + C**2 * u_x**2, x_arr)


# Exact energy (constant for all t)
E_exact = 0.5 * C**2 * (np.pi / L_DOM)**2 * 0.5 * L_DOM  # = π²c²/(4L)

print("=" * 72)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'E (exact':>10}")
print(f"{'':22} {'':>12} {'':>12} {'':>12} {f'= {E_exact:.4f})':>10}")
print("-" * 72)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {E_exact:>10.4f}")
print(f"{'Leapfrog':22} {err_lf_ev.max():>12.3e} {err_lf_ev.mean():>12.3e} "
      f"{l2_norm(err_lf_ev, x_ev):>12.3e} {energy(u_lf_ev, x_ev, T_END):>10.4f}")
print(f"{'Newmark-β':22} {err_nm_ev.max():>12.3e} {err_nm_ev.mean():>12.3e} "
      f"{l2_norm(err_nm_ev, x_ev):>12.3e} {energy(u_nm_ev, x_ev, T_END):>10.4f}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {energy(u_pinn_ev, x_ev, T_END):>10.4f}")
print("=" * 72)
print(f"\nNote: Total energy E = (1/2)∫[u_t² + c²u_x²] dx should be conserved ≈ {E_exact:.4f}.")

---

## Summary

### About the Wave Equation

The wave equation is a cornerstone of mathematical physics and engineering. Key properties:

- It is a **linear, second-order hyperbolic PDE** — fundamentally different from the parabolic equations (heat, Fokker–Planck, Kolmogorov). No diffusion, no energy dissipation.
- Solutions propagate at the **finite speed** $c$ — the **domain of dependence** is bounded by backward characteristics. The CFL condition for explicit schemes directly reflects this physical property.
- **Energy conservation**: the total mechanical energy $E = \frac{1}{2}\int[u_t^2 + c^2 u_x^2]\,dx$ is invariant. Numerical schemes that violate this will exhibit secular drift.
- The problem is **second-order in time**, requiring *two* initial conditions (displacement and velocity) — this distinguishes it from first-order evolution equations.

### Method Comparison

| Aspect | Leapfrog (Explicit) | Newmark-β (Implicit) | PINN |
|--------|-------------------|---------------------|------|
| **Core idea** | Central differences in $x$ and $t$ | Average-acceleration implicit scheme | Minimise PDE + IC + BC residuals |
| **Accuracy** | $\mathcal{O}(\Delta t^2,\,h^2)$ | $\mathcal{O}(\Delta t^2,\,h^2)$ | Depends on training |
| **Stability** | Conditional (CFL: $r \leq 1$) | Unconditional ($\beta \geq 1/4$, $\gamma = 1/2$) | Unconditional |
| **Energy conserv.** | Exact at $r = 1$ | Exact ($\gamma = 1/2$) | Soft (through loss) |
| **Numerical dissipation** | None | None ($\gamma = 1/2$) | None by design |
| **Linear system** | None (explicit) | Tridiagonal solve per step | None (gradient descent) |
| **Mesh required** | Yes — uniform 1D grid | Yes — uniform 1D grid | No — meshfree |
| **Best for** | Simple wave problems, when CFL is acceptable | Large time steps, structural dynamics | High-dimensional or inverse wave problems |

### Key Observations

- The **leapfrog** scheme is the natural explicit method for the wave equation — it is second-order in both space and time, non-dissipative, and trivially simple. At $r = 1$ (CFL limit) it reproduces the *exact* D'Alembert solution on the grid.
- **Newmark-β** ($\beta = 1/4$, $\gamma = 1/2$) is unconditionally stable with *no numerical dissipation*, making it the standard workhorse for structural dynamics applications. It requires a tridiagonal solve per step — negligible cost for 1-D.
- The **PINN** must learn *two* second derivatives ($u_{tt}$ and $u_{xx}$) and enforce *two* initial conditions. This makes the wave equation more challenging for PINNs than parabolic equations — sufficient network capacity and collocation density are critical.